# Fitness prediction GNN — MIF-ST node and edge features

Predicts the fitness of TEM beta-lactamase variants from an AlphaFold2 structure
and a pretrained MIF-ST representation.

Each variant becomes one graph:

- **nodes** — 286 residues, features = MIF-ST representation (256) + CA coordinate (3) = **259**
- **edges** — the k = 5 nearest residues by virtual CB-CB distance, directed
  (neighbour to residue), not symmetrised, no sequential backbone edges:
  286 x 5 = 1430 edges per graph
- **edge features** — distance, omega, theta, phi + the MIF-ST representation of
  the source residue (256) = **260**
- **target** — the variant's fitness

The model is `NNConv(259->128) -> NNConv(128->64) -> NNConv(64->32)`, each with an
edge network `Linear(260, h) -> ReLU -> Linear(h, d_in*d_out)` and `aggr='mean'`,
followed by global mean pooling and `Linear(32, 1)`. MSE loss, Adam,
`batch_size = 1`, 200 epochs, with 30 Optuna trials of 15 epochs to choose the
learning rate, weight decay and dropout.

### Before running

1. Extract the MIF-ST representations with `extract_mifst_clean.ipynb`. One
   `<name>_mifst_per_tok.pt` of shape `[286, 256]` per variant.
2. Set the paths in the **Config** cell below.
3. Run the smoke-test cell before committing to the full job.

Train, validation and test sets are separate files. The validation set is used
for the Optuna objective and for the loss curve; the test set is touched only in
the evaluation cell at the end.

### Notes for anyone reproducing this

- Seeds are set, but results still differ between CPU and GPU: reduction order
  differs and PyTorch Geometric's scatter operations are not deterministic on
  CUDA.
- The reported model is the one at the **final epoch**. The lowest validation
  loss is tracked for reporting, but no checkpoint is restored and no early
  stopping is applied.
- The ablations reported in the paper are small changes to this notebook:
  `x = ca` (no MIF-ST on the nodes), `x = embedded` (no coordinates), dropping the
  three angle blocks from `edge_attr`, pointing `PDB_DIR` at the trRosetta
  structures, or skipping the Optuna cells and using a fixed setting.


## 1. Imports

In [ ]:
import os, gc, time, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm

from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.nn import NNConv, global_mean_pool

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

import optuna
import matplotlib.pyplot as plt
import seaborn as sns

# from the MIF-ST source files in this folder
from pdb_utils import parse_PDB, process_coords

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())


## 2. Configuration

In [ ]:
# ============================== CONFIG ==============================
# Paths are relative to this notebook, which sits next to the MIF-ST source files.
DATA_DIR  = "data"
PDB_DIR   = os.path.join(DATA_DIR, "PDB_AlphaFold") + os.sep 
EMB_DIR   = os.path.join(DATA_DIR, "output_extract_mifst_AlphaFold") + os.sep
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
VAL_CSV   = os.path.join(DATA_DIR, "val.csv")
TEST_CSV  = os.path.join(DATA_DIR, "test.csv")
OUT_DIR   = "results" + os.sep

k          = 5          # spatial neighbours per residue
SEED       = 42
BATCH_SIZE = 1          # 1 is the protocol; larger batches change the optimisation
N_TRIALS   = 30         # Optuna trials
TUNE_EPOCH = 15         # epochs per trial
NUM_EPOCHS = 200        # final training
# ====================================================================

os.makedirs(OUT_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("device =", device)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU    = {p.name}, {p.total_memory/1e9:.1f} GB VRAM")


## 3. Build the graphs


In [ ]:
def build_graphs(csv_path, split_name):
    """One pass per variant: geometry + representation -> PyG Data.

    The same function builds the training, validation and test graphs, so the
    three cannot drift apart.

    ABLATIONS (see the paper): `x = torch.cat([embedded, ca], dim=1)` below is the
    node feature. Use `x = ca` to drop MIF-ST from the nodes while keeping it on
    the edges, or `x = embedded` to drop the coordinates. To drop the angles,
    build `edge_attr` from `[distances, neighbor_emb]` only.
    """
    df = pd.read_csv(csv_path).reset_index()
    data_list = []

    with tqdm(total=len(df), desc=f"Building {split_name} graphs") as pbar:
        for _, row in df.iterrows():
            name, fit = row['name'], row['fitness']

            # --- geometry from the AF2 structure -----------------------------
            coords, wt, _ = parse_PDB(PDB_DIR + row['pdb'])
            coords = {'N': coords[:, 0], 'CA': coords[:, 1], 'C': coords[:, 2]}
            dist, omega, theta, phi = process_coords(coords)   # Cbeta-Cbeta distances

            # --- residue embedding -------------------------------------------
            embedded = torch.load(EMB_DIR + row['embed'])
            if not torch.is_tensor(embedded):
                embedded = torch.as_tensor(embedded)
            embedded = embedded.float()

            ca = torch.tensor(coords['CA'], dtype=torch.float32)
            assert embedded.shape[0] == ca.shape[0], (
                f"{name}: the representation has {embedded.shape[0]} rows but the structure "
                f"has {ca.shape[0]} residues -- were they extracted from this PDB set?"
            )

            x = torch.cat([embedded, ca], dim=1)

            distance = torch.tensor(dist,  dtype=torch.float32)
            omega_t  = torch.tensor(omega, dtype=torch.float32)
            theta_t  = torch.tensor(theta, dtype=torch.float32)
            phi_t    = torch.tensor(phi,   dtype=torch.float32)
            for t in (distance, omega_t, theta_t, phi_t):
                t[torch.isnan(t)] = 0

            sorted_distances, sorted_indices = torch.sort(distance, dim=1)
            distances = sorted_distances[:, 1:k + 1]      # drop self, keep k nearest
            indices   = sorted_indices[:, 1:k + 1]
            omegas = omega_t.gather(1, indices)
            thetas = theta_t.gather(1, indices)
            phis   = phi_t.gather(1, indices)

            source_indices = indices.flatten()
            target_indices = torch.repeat_interleave(torch.arange(len(indices)), k)
            edge_index = torch.stack([source_indices, target_indices], dim=0)  # directed

            neighbor_emb = embedded[edge_index[0]]
            edge_attr = torch.cat([
                distances.flatten().unsqueeze(1),
                omegas.flatten().unsqueeze(1),
                thetas.flatten().unsqueeze(1),
                phis.flatten().unsqueeze(1),
                neighbor_emb,
            ], dim=1)

            y = torch.tensor(fit, dtype=torch.float)
            data_list.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                                  y=y, name=name))          # <-- FIX: name was missing

            del coords, dist, omega, theta, phi, distance, omega_t, theta_t, phi_t
            pbar.update(1)

    gc.collect()
    print(f"{split_name}: {len(data_list)} graphs | node dim {data_list[0].x.shape[1]}"
          f" | edge dim {data_list[0].edge_attr.shape[1]}"
          f" | edges/graph {data_list[0].edge_index.shape[1]}")
    return data_list


In [ ]:
datatrain_list = build_graphs(TRAIN_CSV, "train")
dataval_list   = build_graphs(VAL_CSV,   "val")
datatest_list  = build_graphs(TEST_CSV,  "test")

NODE_DIM = datatrain_list[0].x.shape[1]
EDGE_DIM = datatrain_list[0].edge_attr.shape[1]

# The validation set drives the Optuna objective, so an overlap with train would
# turn it into a training loss, and an overlap with test would mean the
# hyperparameters were chosen on the data being reported.
_names = {s: {d.name for d in dl} for s, dl in
          (("train", datatrain_list), ("val", dataval_list), ("test", datatest_list))}
for _a, _b in (("train", "val"), ("train", "test"), ("val", "test")):
    _shared = _names[_a] & _names[_b]
    assert not _shared, f"{_a} and {_b} share {len(_shared)} variants: {sorted(_shared)[:5]}"
print("train / val / test are disjoint")


In [ ]:
# ---- How much RAM does the dataset take, and how big is conv1? ----------------
def dataset_gb(dl):
    return sum(d.x.numel() * 4 + d.edge_attr.numel() * 4 + d.edge_index.numel() * 8
               for d in dl) / 1e9

E = datatrain_list[0].edge_index.shape[1]
conv1_out_units = NODE_DIM * 128
conv1_params    = 128 * conv1_out_units + conv1_out_units + (EDGE_DIM * 128 + 128) \
                  + NODE_DIM * 128 + 128
conv1_act_gb    = E * conv1_out_units * 4 / 1e9

print(f"node dim {NODE_DIM} | edge dim {EDGE_DIM} | edges/graph {E}")
print(f"train dataset in RAM : {dataset_gb(datatrain_list):6.2f} GB")
print(f"val   dataset in RAM : {dataset_gb(dataval_list):6.2f} GB")
print(f"test  dataset in RAM : {dataset_gb(datatest_list):6.2f} GB")
print(f"conv1 edge-net params: {conv1_params/1e6:6.2f} M")
print(f"conv1 weight tensor per graph [E, 128, {NODE_DIM}]: {conv1_act_gb:.2f} GB "
      f"(roughly x2 for the backward pass)")
print("\nNNConv turns every edge feature vector into a weight matrix, so this tensor "
      "is what dominates memory.\nAt batch_size 1 it is well under 1 GB; it scales "
      "linearly with the batch size.")


## 4. Model


In [ ]:
class ProteinGNN(torch.nn.Module):
    """NNConv x3 -> global mean pool -> Linear(32, 1).

    `input_dim` and `edge_dim` are passed in from the built graphs rather than
    hardcoded, so the feature ablations need no change here.
    """

    def __init__(self, input_dim, edge_dim, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        nn1 = torch.nn.Sequential(
            torch.nn.Linear(edge_dim, 128), torch.nn.ReLU(),
            torch.nn.Linear(128, input_dim * 128))
        self.conv1 = NNConv(input_dim, 128, nn1, aggr='mean')

        nn2 = torch.nn.Sequential(
            torch.nn.Linear(edge_dim, 64), torch.nn.ReLU(),
            torch.nn.Linear(64, 128 * 64))
        self.conv2 = NNConv(128, 64, nn2, aggr='mean')

        nn3 = torch.nn.Sequential(
            torch.nn.Linear(edge_dim, 32), torch.nn.ReLU(),
            torch.nn.Linear(32, 64 * 32))
        self.conv3 = NNConv(64, 32, nn3, aggr='mean')

        self.fc = torch.nn.Linear(32, 1)

    def forward(self, x, edge_index, edge_attr, batch, return_embedding=False):
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index, edge_attr))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv3(x, edge_index, edge_attr))
        x = F.dropout(x, p=self.dropout, training=self.training)

        embedding = global_mean_pool(x, batch)
        if return_embedding:
            return self.fc(embedding), embedding
        return self.fc(embedding)


def n_params(m):
    return sum(p.numel() for p in m.parameters())


train_loader = DataLoader(datatrain_list, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(dataval_list,   batch_size=1, shuffle=False)
test_loader  = DataLoader(datatest_list,  batch_size=1, shuffle=False)
print(f"batches -- train {len(train_loader)} | val {len(val_loader)} | test {len(test_loader)}")

_probe = ProteinGNN(NODE_DIM, EDGE_DIM)
print(f"total model parameters: {n_params(_probe)/1e6:.2f} M "
      f"({n_params(_probe)*4/1e6:.0f} MB fp32)")
del _probe; gc.collect()


In [ ]:
def vram():
    """(allocated GB, reserved GB) -- reserved climbing while allocated is flat
    means the caching allocator is fragmenting."""
    if not torch.cuda.is_available():
        return (0.0, 0.0)
    return (torch.cuda.memory_allocated() / 1e9, torch.cuda.memory_reserved() / 1e9)


def run_epoch(model, loader, optimizer, loss_fn, train=True):
    """One pass. `out.squeeze(-1)` fixes the [1,1] vs [1] shape mismatch."""
    model.train() if train else model.eval()
    total = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            batch = batch.to(device)
            if train:
                optimizer.zero_grad(set_to_none=True)
            out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            loss = loss_fn(out.squeeze(-1), batch.y)          # <-- FIX
            if train:
                loss.backward()
                optimizer.step()
            total += loss.item()
    return total / len(loader)


## 5. Smoke test — run this before the long jobs


In [ ]:
# ---- SMOKE TEST: 3 epochs before committing to the full run -------------------
# If the per-epoch time is already climbing here, or `reserved` VRAM is close to
# the card's capacity, stop rather than waiting for the full run to crawl.
'''
_m = ProteinGNN(NODE_DIM, EDGE_DIM, dropout=0.2).to(device)
_o = torch.optim.Adam(_m.parameters(), lr=1e-4)
_l = torch.nn.MSELoss()

for ep in range(3):
    t0 = time.time()
    tr = run_epoch(_m, train_loader, _o, _l, train=True)
    va = run_epoch(_m, val_loader, None, _l, train=False)
    a, r = vram()
    print(f"[smoke] epoch {ep+1} | train {tr:.4f} | val {va:.4f} | "
          f"{time.time()-t0:6.1f}s | VRAM alloc {a:.2f} / reserved {r:.2f} GB")

del _m, _o, _l
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
'''

## 6. Optuna tuning


In [ ]:
def objective(trial):
    lr           = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    dropout_rate = trial.suggest_float('dropout', 0.1, 0.5)

    model_GNN = ProteinGNN(input_dim=NODE_DIM, edge_dim=EDGE_DIM,
                           dropout=dropout_rate).to(device)
    optimizer = torch.optim.Adam(model_GNN.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn   = torch.nn.MSELoss()

    best_val_loss = float('inf')
    try:
        with tqdm(total=TUNE_EPOCH, desc=f"Trial {trial.number}", leave=False) as pbar:
            for epoch in range(TUNE_EPOCH):
                tr_loss  = run_epoch(model_GNN, train_loader, optimizer, loss_fn, train=True)
                val_loss = run_epoch(model_GNN, val_loader, None, loss_fn, train=False)
                best_val_loss = min(best_val_loss, val_loss)
                pbar.set_postfix({'train': f"{tr_loss:.4f}", 'val': f"{val_loss:.4f}"})
                pbar.update(1)
    finally:
        # Free the trial's model; 30 trials otherwise fragment VRAM.
        del model_GNN, optimizer, loss_fn
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return best_val_loss


In [ ]:
# Run optuna tuning

t0 = time.time()
sampler = optuna.samplers.TPESampler(seed=SEED)          # reproducible search
study   = optuna.create_study(direction='minimize', sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
tune_seconds = time.time() - t0

best_params   = study.best_trial.params
learning_rate = best_params['lr']
weight_decay  = best_params['weight_decay']
dropout       = best_params['dropout']

print(f"\nBest trial value: {study.best_trial.value:.6f}")
print(f"  Learning Rate: {learning_rate}")
print(f"  Weight Decay : {weight_decay}")
print(f"  Dropout      : {dropout}")
print(f"  Tuning time  : {tune_seconds/60:.1f} min")

study.trials_dataframe().to_csv(OUT_DIR + "optuna_trials.csv", index=False)


In [ ]:
# To skip the Optuna cells above
'''
learning_rate = 0.0001
weight_decay  = 0.0001
dropout       = 0.2
#tune_seconds  = float('nan')
'''

## 7. Final training (200 epochs)


In [ ]:
train_losses, val_losses, epoch_times = [], [], []

model_GNN = ProteinGNN(input_dim=NODE_DIM, edge_dim=EDGE_DIM, dropout=dropout).to(device)
optimizer = torch.optim.Adam(model_GNN.parameters(), lr=learning_rate,
                             weight_decay=weight_decay)
loss_fn   = torch.nn.MSELoss()

print("Starting main training run with best parameters:")
print(f"  Learning Rate: {learning_rate}")
print(f"  Weight Decay : {weight_decay}")
print(f"  Dropout      : {dropout}")
print(f"  Parameters   : {n_params(model_GNN)/1e6:.2f} M")
print("start training")

best_val_loss = float('inf')      # tracked for reporting only: no checkpoint is
                                  # restored, so the reported model is the one at
                                  # the FINAL epoch. There is no early stopping.
total_training_start = time.time()

with tqdm(total=NUM_EPOCHS) as pbar:
    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()

        avg_train_loss = run_epoch(model_GNN, train_loader, optimizer, loss_fn, train=True)
        avg_val_loss   = run_epoch(model_GNN, val_loader, None, loss_fn, train=False)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        best_val_loss = min(best_val_loss, avg_val_loss)

        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)
        a, r = vram()
        tqdm.write(f"Epoch {epoch+1:3} | Train Loss: {avg_train_loss:.4f} "
                   f"| Val Loss: {avg_val_loss:.4f} | Time: {epoch_time:6.2f}s "
                   f"| VRAM {a:.2f}/{r:.2f} GB")
        pbar.update(1)

train_seconds = time.time() - total_training_start
print(f"\nBest validation MSE seen: {best_val_loss:.4f}")
print(f"Final-epoch validation MSE: {val_losses[-1]:.4f}   <- this is the reported model")
print(f"Total training time         : {train_seconds/60:.1f} min")
print(f"Epoch time  first 5 / last 5: {np.mean(epoch_times[:5]):.1f}s / "
      f"{np.mean(epoch_times[-5:]):.1f}s")
if np.mean(epoch_times[-5:]) > 1.5 * np.mean(epoch_times[:5]):
    print("WARNING: epochs slowed by >50% -- VRAM is probably spilling to system RAM.")


## 8. Evaluation


In [ ]:
@torch.no_grad()
def predict(loader):
    model_GNN.eval()
    names, preds, actuals = [], [], []
    for batch in loader:
        batch = batch.to(device)
        out = model_GNN(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        nm = batch.name
        if isinstance(nm, (list, tuple)):        # PyG collates str attrs into a list
            nm = nm[0]
        names.append(nm)
        preds.append(out.squeeze(-1).detach().cpu().item())      # <-- FIX
        actuals.append(batch.y.item())
    return names, preds, actuals


train_names, x_train, actual_train_list = predict(train_loader)
val_names,   x_val,   actual_val_list    = predict(val_loader)
name_list,   x_test,  x_actual           = predict(test_loader)


def report(name, y_true, y_pred):
    print(f"--- {name} ---")
    print(f"Pearson: {pearsonr(y_true, y_pred)[0]:.4f}\n")


report("Train", actual_train_list, x_train)
report("Val",   actual_val_list,   x_val)
report("Test",  x_actual,          x_test)


In [ ]:
epochs = list(range(1, len(train_losses) + 1))

plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.scatterplot(x=actual_train_list, y=x_train, color='blue', alpha=0.5, s=60, label='Train')
sns.scatterplot(x=x_actual, y=x_test, color='red', alpha=0.7, s=60, label='Test')
lo = min(x_actual + actual_train_list); hi = max(x_actual + actual_train_list)
plt.plot([lo, hi], [lo, hi], color='green', linewidth=2, label='y = x')
plt.xlabel('Actual fitness'); plt.ylabel('Predicted fitness')
plt.title('Predicted vs actual'); plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, train_losses, color='red', label='Train loss')
plt.plot(epochs, val_losses, color='green', label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.title('Loss'); plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs, epoch_times, color='purple')
plt.xlabel('Epoch'); plt.ylabel('Seconds'); plt.title('Time per epoch')
plt.axhline(np.mean(epoch_times[:5]), color='gray', linestyle='--',
            label='mean of first 5'); plt.legend()

plt.tight_layout()
plt.savefig(OUT_DIR + "diagnostics.png", dpi=150)
plt.show()


In [ ]:
pd.DataFrame({'name': name_list, 'predicted': x_test, 'actual': x_actual}).to_csv(
    OUT_DIR + "predictions_test.csv", index=False)
pd.DataFrame({'name': val_names, 'predicted': x_val, 'actual': actual_val_list}).to_csv(
    OUT_DIR + "predictions_val.csv", index=False)
pd.DataFrame({'name': train_names, 'predicted': x_train, 'actual': actual_train_list}).to_csv(
    OUT_DIR + "predictions_train.csv", index=False)
pd.DataFrame({'epoch': epochs, 'train_loss': train_losses,
              'val_loss': val_losses, 'seconds': epoch_times}).to_csv(
    OUT_DIR + "loss_curve.csv", index=False)